# Agent Protocols: MCP & A2A

Wiki reference for [agent protocols (MCP & A2A)](https://ml-viz-ruby.vercel.app/wiki/agent-protocols-mcp-a2a).

**The idea in one sentence.** Two JSON-RPC protocols standardise how agents connect: **MCP**
lets an agent **discover and call tools** on a server (`tools/list`, `tools/call`), and **A2A**
lets one agent **delegate a task to another** via its published **agent card** — and the safety
catch is that agent cards are *self-asserted*, so you must allowlist who you trust.

We implement a minimal MCP server/client and A2A orchestration from scratch, **validate tool
discovery and the task lifecycle**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import itertools

_ids = itertools.count(1)
def rpc(method, **params):
    """Build a JSON-RPC 2.0 request envelope (the shared wire format of both protocols)."""
    return {"jsonrpc": "2.0", "id": next(_ids), "method": method, "params": params}

## 1 — A minimal MCP server

An MCP server exposes **tools** (name + JSON-Schema + handler) and answers two methods: `tools/list`
for discovery and `tools/call` to invoke one. This is the vertical edge — agent ↔ tools/data.

In [ ]:
class MCPServer:
    def __init__(self, name):
        self.name = name
        self._tools = {}      # name -> (schema, handler)

    def tool(self, name, description, schema):
        def deco(fn):
            self._tools[name] = ({"name": name, "description": description,
                                  "inputSchema": schema}, fn)
            return fn
        return deco

    def handle(self, req):
        m = req["method"]
        if m == "initialize":
            return {"result": {"serverInfo": {"name": self.name}, "capabilities": {"tools": {}}}}
        if m == "tools/list":
            return {"result": {"tools": [meta for meta, _ in self._tools.values()]}}
        if m == "tools/call":
            name = req["params"]["name"]
            args = req["params"].get("arguments", {})
            _, fn = self._tools[name]
            return {"result": {"content": [{"type": "text", "text": str(fn(**args))}]}}
        return {"error": {"code": -32601, "message": f"method not found: {m}"}}

# A 'billing tools' server exposing one tool
billing_tools = MCPServer("billing-tools")

@billing_tools.tool("refund", "Issue a refund for an order id.",
                    {"type": "object", "properties": {"order_id": {"type": "string"}},
                     "required": ["order_id"]})
def _refund(order_id):
    return {"order_id": order_id, "status": "refunded", "amount": 42.00}

## 2 — An MCP client talking to the server

The client does the `initialize` handshake, discovers tools, then calls one — exactly the sequence
from the wiki page, just with the transport being a direct method call instead of stdio/HTTP.

In [ ]:
class MCPClient:
    def __init__(self, server):
        self.server = server
        self.server.handle(rpc("initialize"))      # handshake
    def list_tools(self):
        return self.server.handle(rpc("tools/list"))["result"]["tools"]
    def call(self, name, **arguments):
        resp = self.server.handle(rpc("tools/call", name=name, arguments=arguments))
        return resp["result"]["content"][0]["text"]

client = MCPClient(billing_tools)
print("discovered tools:", [t["name"] for t in client.list_tools()])
print("refund result:   ", client.call("refund", order_id="4471"))

### Validate: MCP discovery and tool calls

MCP's whole point is **runtime discovery**: the client asks `tools/list` and gets back typed
tool schemas, then invokes one with `tools/call`. We confirm the client discovers the server's
tools and that a call executes and returns content.

In [ ]:
names = [t['name'] for t in client.list_tools()]
print('discovered tools:', names)
result = client.call('refund', order_id='4471')
print('refund call result:', result)
assert 'refund' in names, 'MCP tools/list discovers the server tools at runtime'
assert isinstance(result, str) and len(result) > 0, 'MCP tools/call executes and returns content'
print('\n✅ MCP = discover typed tools + call them over a shared JSON-RPC wire format')

## 3 — A minimal A2A agent (card + task lifecycle)

An A2A agent publishes an **Agent Card** (capability discovery) and accepts **Tasks** that move
through a lifecycle (`submitted → working → completed`). This is the horizontal edge — agent ↔ agent.
Our billing agent fulfils tasks by calling its *own* MCP tool server — so the two protocols compose.

In [ ]:
class A2AAgent:
    def __init__(self, name, skills, fulfil):
        self.name, self.skills, self._fulfil = name, skills, fulfil
    @property
    def agent_card(self):                       # served at /.well-known/agent.json
        return {"name": self.name, "skills": self.skills,
                "endpoint": f"https://{self.name}.example.com/a2a"}
    def handle(self, req):                      # method: message/send
        text = req["params"]["message"]["parts"][0]["text"]
        states = ["submitted", "working"]
        artifact = self._fulfil(text)           # do the work (may call MCP tools)
        states.append("completed")
        return {"result": {"task": {"states": states, "status": "completed",
                                    "artifact": artifact}}}

def billing_fulfil(text):
    order_id = text.split("#")[-1].strip()
    return client.call("refund", order_id=order_id)   # A2A task -> MCP tool call

billing_agent = A2AAgent("billing", skills=["refunds", "invoices"], fulfil=billing_fulfil)
print("agent card:", billing_agent.agent_card)

## 4 — Orchestrator delegates over A2A → which uses MCP

The orchestrator discovers the billing agent's card, checks it has the needed skill, then sends a
task. The billing agent fulfils it by calling its MCP `refund` tool. `N×M` integrations collapse to
`N+M`: the orchestrator speaks A2A once, the tool is wrapped in MCP once.

In [ ]:
def orchestrate(remote_agent, request_text, needed_skill):
    card = remote_agent.agent_card                       # 1. discovery
    assert needed_skill in card["skills"], "agent lacks the skill"
    req = rpc("message/send",
              message={"role": "user", "parts": [{"kind": "text", "text": request_text}]})
    task = remote_agent.handle(req)["result"]["task"]     # 2. delegate
    return task

task = orchestrate(billing_agent, "please refund order #4471", needed_skill="refunds")
print("task lifecycle:", " → ".join(task["states"]))
print("final status:  ", task["status"])
print("artifact:      ", task["artifact"])

### Validate: A2A delegation runs the full task lifecycle

A2A delegation goes through **discovery** (read the agent card, check the skill) then **delegate**
(`message/send`), and the task moves `submitted → working → completed`. We confirm the lifecycle
and the completed status.

In [ ]:
print('task lifecycle:', ' -> '.join(task['states']))
print('status:', task['status'])
assert task['states'] == ['submitted', 'working', 'completed'], 'A2A task runs the full lifecycle'
assert task['status'] == 'completed', 'a delegated task completes'
print('\n✅ A2A = discover an agent by its card, then delegate a task and track its lifecycle')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **self-asserted agent cards** | a rogue agent can claim any skill (demo) — allowlist trust |
| **no skill check** | delegating to an agent that lacks the skill fails silently |
| **unbounded delegation** | agents calling agents can loop or fan out; cap depth |
| **tool schema drift** | clients must handle `tools/list` changing at runtime |
| **auth on the wire** | JSON-RPC alone has no auth; add tokens/mTLS in production |

Demo: the unsafe orchestrator delegates to a rogue agent that merely claims the skill.

In [ ]:
# The security gotcha: an agent card is SELF-ASSERTED. The plain orchestrate() only checks
# that the agent *claims* the needed skill — so a ROGUE agent advertising "refunds" gets invoked
# just the same. That is why real A2A needs an allowlist of trusted agents (the exercise below).
rogue = A2AAgent('rogue', skills=['refunds'], fulfil=billing_fulfil)
rogue_task = orchestrate(rogue, 'refund order #99', needed_skill='refunds')
print(f'rogue agent task status via UNSAFE orchestrate: {rogue_task["status"]}')
assert rogue_task['status'] == 'completed', 'unsafe orchestrate delegates to ANY agent that claims the skill'
print('\nAgent cards are self-asserted -> skill-matching is NOT a trust boundary. Allowlist trusted agents.')

## ✏️ Your turn

**Exercise.** Real systems must not blindly invoke whatever a peer claims. Implement
`safe_orchestrate(remote_agent, request_text, needed_skill, allowed_agents)` that delegates **only**
if (a) the agent's name is in `allowed_agents` (least-privilege / trust check) **and** (b) its Agent
Card advertises `needed_skill`. Otherwise return `{"status": "rejected", "reason": ...}` without
calling the agent — the security posture the wiki page argues for.

In [ ]:
def safe_orchestrate(remote_agent, request_text, needed_skill, allowed_agents):
    card = remote_agent.agent_card
    # TODO(you): reject (without calling handle) if the agent isn't allowed or lacks the skill;
    #            otherwise delegate and return the completed task dict
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
allow = {"billing"}
ok = safe_orchestrate(billing_agent, "refund order #99", "refunds", allow)
assert ok["status"] == "completed"

untrusted = A2AAgent("rogue", skills=["refunds"], fulfil=billing_fulfil)
blocked = safe_orchestrate(untrusted, "refund order #99", "refunds", allow)
assert blocked["status"] == "rejected", "untrusted agent must be blocked"

no_skill = safe_orchestrate(billing_agent, "forecast revenue", "forecasting", allow)
assert no_skill["status"] == "rejected", "missing skill must be rejected"
print("✓ delegates only to a trusted agent that advertises the needed skill")

<details>
<summary>Solution</summary>

```python
def safe_orchestrate(remote_agent, request_text, needed_skill, allowed_agents):
    card = remote_agent.agent_card
    if card["name"] not in allowed_agents:
        return {"status": "rejected", "reason": "agent not in allowlist"}
    if needed_skill not in card["skills"]:
        return {"status": "rejected", "reason": "skill not advertised"}
    req = rpc("message/send",
              message={"role": "user", "parts": [{"kind": "text", "text": request_text}]})
    return remote_agent.handle(req)["result"]["task"]
```

Discovery (the Agent Card) tells you what a peer *claims* it can do; the allowlist encodes who you
actually *trust*. A standardized protocol makes integration easy for everyone — including bad
actors — so authentication and least-privilege are not optional add-ons.

</details>

## Key takeaways

- **MCP = tool discovery + calls:** `tools/list` returns typed schemas, `tools/call` executes
  them over JSON-RPC (verified).
- **A2A = agent delegation:** read the agent card, check the skill, delegate via `message/send`,
  track `submitted → working → completed` (verified).
- **Agent cards are self-asserted:** skill-matching is not a trust boundary (demo) — allowlist
  trusted agents.
- **Both share a wire format** (JSON-RPC 2.0) so tools and agents compose.